# OASIS-1 174-Subject Multitask Evaluation

This notebook summarizes the larger public-data evaluation for the single-MRI multitask pipeline:

```text
Single T1 MRI
-> anatomical representation / segmentation
-> dementia classification
-> cognitive prediction
```

The available balanced OASIS-1 subset with usable MRI, MMSE, and CDR labels contained **174 subjects**.


## UCSF Paper vs This Public-Data Reproduction

| Area | UCSF paper | This project |
| --- | --- | --- |
| Overall goal | Predict categorical and continuous Alzheimer's disease outcomes from a single MRI scan | Reproduce the same modeling strategy using public substitute data |
| MRI input | Single T1-weighted MRI scan | Single T1-weighted MRI scan |
| Main data source | ADNI and HCP-YA style data resources | OASIS-1 public dataset with 174 usable subjects |
| Anatomical representation | Learns brain anatomy features as part of the modeling strategy | Uses FSL-FAST tissue labels and a 3D multitask model to learn anatomical segmentation features |
| Disease prediction | Predicts Alzheimer's-related diagnostic categories | Predicts a simplified dementia label using `CDR 0` vs `CDR > 0` |
| Cognitive prediction | Predicts continuous cognitive outcomes such as ADAS-Cog style measures | Predicts MMSE as the available cognitive score |
| Training scale | Research-scale data, preprocessing, compute, and validation | CPU-friendly reproduction scaffold with a basic train/validation/test split |
| Current interpretation | Published study with reported clinical prediction performance | End-to-end technical reproduction; segmentation is strong, while clinical prediction remains preliminary |


## Experiment Setup

- Dataset: OASIS-1 public substitute for ADNI
- Subjects: 174 total
- Split: 121 train / 26 validation / 27 test
- MRI input: one downsampled T1 MRI per subject
- Segmentation target: FSL-FAST tissue labels
- Dementia label: `CDR 0` vs `CDR > 0`
- Cognitive target: MMSE
- Model: 3D multitask UNet-style model


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

PROJECT = Path.cwd()
if not (PROJECT / "outputs").exists():
    PROJECT = PROJECT.parent

RESULT_DIR = PROJECT / "outputs" / "oasis1_200_eval"
PREDICTIONS = RESULT_DIR / "predictions.csv"
METRICS = RESULT_DIR / "metrics.csv"
CONFUSION = RESULT_DIR / "confusion_matrix.png"
SLICES = RESULT_DIR / "mri_fast_model_slices.png"

metrics = pd.read_csv(METRICS)
predictions = pd.read_csv(PREDICTIONS)


## Metrics By Split

Balanced accuracy is reported alongside raw accuracy because it better reflects performance when class predictions are uneven.


In [ ]:
display(metrics)


## Main Test-Set Result

For the held-out test split:

| Task | Metric | Result | Interpretation |
| --- | ---: | ---: | --- |
| Anatomical segmentation | Dice | 0.884 | Tissue segmentation performance |
| Dementia classification | Accuracy | 0.519 | Thresholded classification accuracy |
| Dementia classification | Balanced accuracy | 0.500 | Limited class separation in this run |
| Dementia classification | AUC | 0.758 | Preliminary probability-ranking signal |
| Cognitive prediction | MMSE MAE | 2.62 | Baseline cognitive prediction result |
| Cognitive prediction | MMSE RMSE | 3.37 | Moderate average error |


## Prediction Table

Each row contains the subject ID, split, true clinical label, predicted dementia probability, true MMSE, predicted MMSE, and segmentation Dice.


In [ ]:
display(predictions.head(20))
print(f"Rows: {len(predictions)}")
print(predictions.groupby(["split", "true_label"]).size())


## Confusion Matrix

The confusion matrix shows the thresholded classification behavior. At the default 0.5 threshold, the model predicted every subject as control/non-demented.

![Confusion matrix](../outputs/oasis1_200_eval/confusion_matrix.png)


In [ ]:
display(Image(filename=str(CONFUSION)))


## MRI / FAST / Model Segmentation Slices

These visual outputs show the anatomical part of the pipeline:

1. T1 MRI slice
2. FSL-FAST tissue target
3. Model-predicted segmentation

![MRI FAST model slices](../outputs/oasis1_200_eval/mri_fast_model_slices.png)


In [ ]:
display(Image(filename=str(SLICES)))


## Interpretation

The larger OASIS run demonstrates the end-to-end technical pipeline:

```text
MRI loading
-> FAST segmentation targets
-> 3D multitask model
-> segmentation output
-> dementia probability output
-> MMSE output
-> evaluation tables and images
```

The anatomical segmentation output achieved test Dice around **0.884**.

The dementia classifier remained limited in this run. Predicted probabilities stayed near **0.494**, just below the 0.5 threshold, so every subject was classified as control. This produced chance-level balanced accuracy.

The MMSE head predicted near the cohort mean, giving test MAE around **2.62**.
